# 03 – Statistiques finales du dataset

Analyse du dataset final anonymisé et splitté, prêt pour SFT et DPO.

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path('..')))

SFT_DIR  = Path('../data/processed/sft')
DPO_DIR  = Path('../data/processed/dpo')
EVAL_DIR = Path('../data/processed/eval_clinique')
META     = Path('../data/metadata/dataset_card.json')

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f]

sft  = {s: load_jsonl(SFT_DIR / f'{s}.jsonl') for s in ['train', 'val', 'test']}
dpo  = {s: load_jsonl(DPO_DIR / f'{s}.jsonl') for s in ['train', 'val', 'test']}
eval_clin = load_jsonl(EVAL_DIR / 'eval_clinique.jsonl')

print('Dataset chargé.')

## 1. Vue d'ensemble

In [ ]:
print('=== Dataset SFT ===')
total_sft = sum(len(v) for v in sft.values())
for split, records in sft.items():
    print(f'  {split:6s} : {len(records):>5} ({len(records)/total_sft*100:.1f}%)')
print(f'  {'TOTAL':6s} : {total_sft}')

print('\n=== Dataset DPO ===')
total_dpo = sum(len(v) for v in dpo.values())
for split, records in dpo.items():
    print(f'  {split:6s} : {len(records):>5} ({len(records)/total_dpo*100:.1f}%)')
print(f'  {'TOTAL':6s} : {total_dpo}')

print(f'\n=== Eval clinique ===')
print(f'  eval   : {len(eval_clin):>5} (séparé strict — jamais vu en entraînement)')

## 2. Distribution par source (SFT train)

In [ ]:
source_counts = Counter(r['source'] for r in sft['train'])
total = sum(source_counts.values())
print('Distribution par source (SFT train) :')
for source, count in source_counts.most_common():
    print(f'  {source:<30s} : {count:>5} ({count/total*100:.1f}%)')

## 3. Distribution par langue

In [ ]:
all_sft = sft['train'] + sft['val'] + sft['test']
lang_counts = Counter(r['language'] for r in all_sft)
total = sum(lang_counts.values())
print('Distribution par langue (SFT total) :')
for lang, count in lang_counts.most_common():
    print(f'  {lang} : {count:>5} ({count/total*100:.1f}%)')

## 4. Longueur des instructions et réponses (SFT)

In [ ]:
def wc(text):
    return len(str(text).split())

train_records = sft['train']
instr_lens = [wc(r['instruction']) for r in train_records]
resp_lens  = [wc(r['response'])    for r in train_records]

print('Longueurs (en mots) — SFT train :')
print(f'  Instruction — min: {min(instr_lens)}, moy: {sum(instr_lens)//len(instr_lens)}, max: {max(instr_lens)}')
print(f'  Réponse     — min: {min(resp_lens)},  moy: {sum(resp_lens)//len(resp_lens)},  max: {max(resp_lens)}')

# Distribution par tranches
buckets_i = Counter()
for l in instr_lens:
    if l <= 20:   buckets_i['0-20'] += 1
    elif l <= 50: buckets_i['21-50'] += 1
    elif l <= 150: buckets_i['51-150'] += 1
    elif l <= 500: buckets_i['151-500'] += 1
    else:          buckets_i['500+'] += 1

print('\nDistribution longueur instruction :')
for bucket, count in sorted(buckets_i.items()):
    print(f'  {bucket:10s} : {count:>5} ({count/len(instr_lens)*100:.1f}%)')

## 5. Qualité DPO — écarts de scores chosen/rejected

In [ ]:
dpo_train = dpo['train']
gaps = [
    r['metadata']['chosen_score'] - r['metadata']['rejected_score']
    for r in dpo_train
    if 'chosen_score' in r.get('metadata', {})
]

if gaps:
    print(f'Écart chosen/rejected (DPO train) :')
    print(f'  min : {min(gaps):.2f}')
    print(f'  moy : {sum(gaps)/len(gaps):.2f}')
    print(f'  max : {max(gaps):.2f}')
    
    # Vérification : aucune paire avec écart nul
    zero_gap = sum(1 for g in gaps if g == 0)
    print(f'\nPaires avec écart = 0 : {zero_gap} (attendu : 0)')
else:
    print('Scores non disponibles dans les métadonnées.')

## 6. Vérification de la séparation eval_clinique

In [ ]:
eval_ids = {r['id'] for r in eval_clin}
train_ids = {r['id'] for r in sft['train']}
overlap = eval_ids & train_ids

print(f'IDs eval_clinique : {len(eval_ids)}')
print(f'IDs SFT train     : {len(train_ids)}')
print(f'Overlap           : {len(overlap)} (attendu : 0)')

if len(overlap) == 0:
    print('\n✅ Séparation stricte confirmée — aucune fuite entre eval et train.')
else:
    print(f'\n❌ ATTENTION : {len(overlap)} enregistrements en commun !')

## 7. Aperçu du dataset_card.json

In [ ]:
with open(META, encoding='utf-8') as f:
    card = json.load(f)

print(json.dumps(card, ensure_ascii=False, indent=2))

## 8. Conclusion

Le dataset est prêt pour la semaine 2 (SFT) :

| Check | Statut |
|---|---|
| 0 erreur de validation | ✅ |
| Séparation eval_clinique / train | ✅ |
| Anonymisation Presidio appliquée | ✅ |
| Format JSONL unifié SFT + DPO | ✅ |
| Log RGPD complet | ✅ |
| Bilingue FR/EN | ✅ |